In [ ]:
from pathlib import Path
import os, glob

HF_HOME = "/media/pc1/Ubuntu/Extend_Data/ngoc/hf"

repo_cache = Path(HF_HOME) / "hub" / "models--mistralai--Mistral-7B-Instruct-v0.1" / "snapshots"
candidates = sorted(repo_cache.glob("*"))
print("Snapshots:", [str(p) for p in candidates])

# pick the newest snapshot that has the shards
local_snapshot = None
for p in reversed(candidates):
    if glob.glob(str(p / "model-*.safetensors")):
        local_snapshot = str(p)
        break

print("Using snapshot:", local_snapshot)
assert local_snapshot, "No local snapshot with model-*.safetensors found."


Snapshots: []
Using snapshot: None


AssertionError: No local snapshot with model-*.safetensors found.

In [1]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

"""
Merged Leishmania RAG Pipeline
- Stage 1 (fast): parallel text+image extraction with robust fallbacks and file-hash tracking
- Stage 2 (smart): semantic chunking + (deterministic) LLM enrichment + tiering + Q&A + summary + multimodal pairs
"""

import os
import re
import json
import time
import math
import hashlib
import logging
import shutil
import tempfile
from dataclasses import dataclass
from datetime import datetime
from pathlib import Path
from typing import List, Dict, Tuple, Optional, Set, Any

# ------------------
# Third-party deps
# ------------------
import pandas as pd
import numpy as np
import PyPDF2
import fitz  # PyMuPDF
import multiprocessing as mp
from concurrent.futures import ProcessPoolExecutor, as_completed

# Semantic & NLP
import nltk
from nltk.tokenize import sent_tokenize, word_tokenize
from nltk.corpus import stopwords
from ast import literal_eval

import spacy

from sentence_transformers import SentenceTransformer
from sklearn.cluster import AgglomerativeClustering

# HF Transformers
import torch
from transformers import (
    AutoModel,
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    pipeline as hf_pipeline,
)

# ------------------
# Global Config
# ------------------

# (Optional) HF cache paths
os.environ["HF_HOME"]       = "/data4t/hf"
os.environ["HF_HUB_CACHE"]  = "/data4t/hf/hub"
os.environ["TRANSFORMERS_CACHE"] = "/data4t/hf/transformers"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
# --- OFFLINE + CACHE HARDENING ---
HF_ROOT = Path("/data4t/hf")

# Make sure both old and new hub cache envs are set
os.environ["HF_HOME"] = str(HF_ROOT)
os.environ["TRANSFORMERS_CACHE"] = str(HF_ROOT / "transformers")
os.environ["HF_HUB_CACHE"] = str(HF_ROOT / "hub")
os.environ["HUGGINGFACE_HUB_CACHE"] = os.environ["HF_HUB_CACHE"]  # older clients use this
os.environ["HF_DATASETS_CACHE"] = str(HF_ROOT / "datasets")

# Optional: force offline (set to "1" if you truly have no internet)
os.environ.setdefault("TRANSFORMERS_OFFLINE", "1")
os.environ.setdefault("HF_HUB_OFFLINE", "1")
os.environ.setdefault("HF_DATASETS_OFFLINE", "1")

def _repo_cache_roots():
    """Return both cache layouts used by different HF versions."""
    return [
        Path(os.environ["TRANSFORMERS_CACHE"]),               # .../transformers/models--{org}--{repo}/...
        Path(os.environ["HF_HUB_CACHE"]),                     # .../hub/models--{org}--{repo}/...
    ]

# Optional: expand common short names → org/name
_DEFAULT_ORG = {
    "all-MiniLM-L6-v2": ("sentence-transformers", "all-MiniLM-L6-v2"),
    "Mistral-7B-Instruct-v0.1": ("mistralai", "Mistral-7B-Instruct-v0.1"),
    "Mixtral-8x7B-Instruct-v0.1": ("mistralai", "Mixtral-8x7B-Instruct-v0.1"),
}

def normalize_repo_id(repo_or_path: str) -> str:
    """Convert various input formats to standard org/name format."""
    p = Path(repo_or_path)
    if p.exists():               # absolute path provided
        return str(p)
    if "/" in repo_or_path:      # already org/name
        return repo_or_path
    if repo_or_path.startswith("models--"):  # cache dir pattern
        # models--org--name -> org/name
        m = re.match(r"models--([^/\\-]+)--(.+)$", repo_or_path)
        if m:
            org, name = m.group(1), m.group(2)
            return f"{org}/{name}"
    if repo_or_path in _DEFAULT_ORG:
        org, name = _DEFAULT_ORG[repo_or_path]
        return f"{org}/{name}"
    # last resort: assume sentence-transformers for unknown bare ids
    return f"sentence-transformers/{repo_or_path}"

def resolve_local_hf_repo(repo_or_path: str, revision: str = "main") -> Optional[Path]:
    """
    Robust resolver that returns the absolute path to the snapshot directory.
    Handles short names, org/name, cache dir names, and direct paths.
    """
    # If a direct path pointing inside snapshots was given, honor it.
    direct = Path(repo_or_path)
    if direct.exists():
        # If it's already a snapshots dir, return it
        if re.search(r"models--[^/\\]+--[^/\\]+[/\\]snapshots[/\\][0-9a-f]{6,}$", str(direct)):
            return direct
        # If it's a model root dir, dive into snapshots
        for root in _repo_cache_roots():
            if str(root) in str(direct):
                snaps_root = direct / "snapshots"
                if snaps_root.exists():
                    snaps = [p for p in snaps_root.iterdir() if p.is_dir()]
                    if snaps:
                        return max(snaps, key=lambda p: p.stat().st_mtime)

    rid = normalize_repo_id(repo_or_path)
    if "/" not in rid:
        return None
    org, name = rid.split("/", 1)
    repo_dir_name = f"models--{org}--{name}"

    # Look for exact repo dir first
    for root in _repo_cache_roots():
        base = root / repo_dir_name
        if not base.exists():
            continue

        # Try refs/<revision>
        refs = base / "refs" / revision
        if refs.exists():
            try:
                sha = refs.read_text().strip()
                snap = base / "snapshots" / sha
                if snap.exists():
                    return snap
            except Exception:
                pass

        # Fallback: pick newest snapshot by mtime
        snaps_root = base / "snapshots"
        if snaps_root.exists():
            snaps = [p for p in snaps_root.iterdir() if p.is_dir()]
            if snaps:
                return max(snaps, key=lambda p: p.stat().st_mtime)

    return None

class TransformersEmbedder:
    """Drop-in replacement if SentenceTransformer files are incomplete offline."""
    def __init__(self, path_or_repo: str, device: Optional[str] = None):
        snap = resolve_local_hf_repo(path_or_repo)
        lp = str(snap) if snap else normalize_repo_id(path_or_repo)
        self.tokenizer = AutoTokenizer.from_pretrained(
            lp, cache_dir=os.environ["TRANSFORMERS_CACHE"], local_files_only=True
        )
        self.model = AutoModel.from_pretrained(
            lp, cache_dir=os.environ["TRANSFORMERS_CACHE"],
            local_files_only=True, torch_dtype=torch.float16 if torch.cuda.is_available() else None
        )
        self.device = device or ("cuda" if torch.cuda.is_available() else "cpu")
        self.model.to(self.device)
        self.model.eval()

    @torch.no_grad()
    def encode(self, sentences: List[str], batch_size: int = 64, show_progress_bar: bool=False):
        import numpy as np
        out = []
        for i in range(0, len(sentences), batch_size):
            batch = sentences[i:i+batch_size]
            enc = self.tokenizer(batch, padding=True, truncation=True, return_tensors="pt")
            enc = {k: v.to(self.device) for k, v in enc.items()}
            outputs = self.model(**enc)
            # mean pooling
            attn = enc["attention_mask"].unsqueeze(-1)  # (B, T, 1)
            last = outputs.last_hidden_state            # (B, T, H)
            summed = (last * attn).sum(dim=1)
            denom = attn.sum(dim=1).clamp(min=1e-9)
            emb = (summed / denom).detach().cpu().float().numpy()
            out.append(emb)
        return np.vstack(out) if out else np.zeros((0, self.model.config.hidden_size), dtype="float32")

def load_sentence_embedder(name_or_repo: str):
    snap = resolve_local_hf_repo(name_or_repo)
    target = str(snap) if snap else normalize_repo_id(name_or_repo)
    try:
        # Try the true SentenceTransformer (requires modules.json etc.)
        return SentenceTransformer(target,
                                   cache_folder=os.environ["TRANSFORMERS_CACHE"],
                                   local_files_only=True)
    except Exception as e:
        logger.warning(f"SentenceTransformer load failed ({e}); using TransformersEmbedder fallback.")
        return TransformersEmbedder(target)

def load_llm(repo_id: str):
    snap = resolve_local_hf_repo(repo_id)
    path = str(snap) if snap else normalize_repo_id(repo_id)
    logger.info(f"LLM loading from: {path}")
    bnb = BitsAndBytesConfig(
        load_in_4bit=True, bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.float16, bnb_4bit_use_double_quant=True
    )
    tok = AutoTokenizer.from_pretrained(
        path, use_fast=True, cache_dir=os.environ["TRANSFORMERS_CACHE"], local_files_only=True
    )
    model = AutoModelForCausalLM.from_pretrained(
        path,
        quantization_config=bnb,
        device_map="auto",
        torch_dtype=torch.float16,
        trust_remote_code=True,
        cache_dir=os.environ["TRANSFORMERS_CACHE"],
        local_files_only=True,
        low_cpu_mem_usage=True,
    )
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.set_float32_matmul_precision("high")
    return hf_pipeline(
        "text-generation",
        model=model, tokenizer=tok,
        max_new_tokens=512, do_sample=False,
        return_full_text=False, device_map="auto"
    )

# I/O
TEXTBOOK_SOURCE_DIR        = Path("data/all_leishmania_sources")
RAG_OUTPUT_DIR             = Path("kaggle/working_v2/rag_knowledge_base")
FINETUNE_OUTPUT_DIR        = Path("kaggle/working_v2/fine_tuning_data")
PROCESSED_METADATA_PATH    = Path("kaggle/working_v2/textbook_processing_metadata.csv")
CHECKPOINT_PATH            = Path("kaggle/working_v2/processing_checkpoint.json")
FILE_HASHES_PATH           = Path("kaggle/working_v2/file_hashes.json")
# --- Shared with generating answer ---
PAGE_RENDERS_DIR       = RAG_OUTPUT_DIR / "page_renders"   # ColQwen2 expects this
META_DIR               = RAG_OUTPUT_DIR / "metadata"       # *_images.json lives here
EVIDENCE_UNITS_DIR     = RAG_OUTPUT_DIR / "evidence_units" # optional but boosts reranking

# Prefilter (version-safe search_for used below; no hit_max kwarg)
HUGE_PDF_PAGES_THRESHOLD   = 800
HUGE_PDF_BYTES_THRESHOLD   = 150 * 1024 * 1024  # 150 MB
PREFILTER_KEYWORDS         = [
    "leishmania", "leishmaniasis", "visceral leishmaniasis", "cutaneous leishmaniasis",
    "post-kala-azar", "kala-azar", "phlebotomine", "sandfly", "phlebotomus", "lutzomyia"
]
PREFILTER_CONTEXT_PAGES    = 2
PREFILTER_MAX_HITS         = 50

# Semantic chunking
EMBEDDING_MODEL_NAME       = "sentence-transformers/all-MiniLM-L6-v2"
MIN_CHUNK_SIZE             = 120
MAX_CHUNK_SIZE             = 900
CHUNK_OVERLAP              = 100  # not used in clustering path, kept for future
DISABLE_PROGRESS_BAR       = True

# Enrichment model (choose one)
# LLM_MODEL_NAME           = "mistralai/Mixtral-8x7B-Instruct-v0.1"   # heavier, MoE
LLM_MODEL_NAME             = "mistralai/Mistral-7B-Instruct-v0.1"     # default (VRAM-friendly)

# Enrichment runtime + thresholds
BATCH_SIZE_GENERATION      = 6
MAX_ENRICH_SECONDS_PER_CHUNK = 15
MIN_RELEVANCE_SCORE        = 0.5   # ↑ stricter than 0.4
HIGH_RELEVANCE_THRESHOLD   = 0.8

# Parallel extraction
MAX_WORKERS                = min(4, mp.cpu_count())
EXTRACTION_CHUNK_SIZE      = 1  # one file per sub-process at a time

# Logging
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s - %(processName)s - %(levelname)s - %(message)s"
)
logger = logging.getLogger("leish-pipeline")

# ------------------
# Utilities
# ------------------

def ensure_nltk():
    """Robust NLTK availability including punkt_tab for newer NLTK."""
    packages = [
        ("tokenizers/punkt", "punkt"),
        ("tokenizers/punkt_tab", "punkt_tab"),
        ("corpora/stopwords", "stopwords"),
    ]
    for finder, pkg in packages:
        try:
            nltk.data.find(finder)
        except LookupError:
            nltk.download(pkg, quiet=True)

ensure_nltk()

def ensure_spacy_model():
    """Load spaCy small English, download if missing."""
    try:
        import spacy  # use global/module import to avoid local shadowing
        return spacy.load("en_core_web_sm")
    except OSError:
        try:
            from spacy.cli import download  # import only the function to avoid rebinding 'spacy'
            download("en_core_web_sm")
            import spacy
            return spacy.load("en_core_web_sm")
        except Exception as e:
            logger.warning(f"spaCy unavailable, fallback to NLTK. Reason: {e}")
            return None

# ------------------
# Hash tracking
# ------------------

class FileHashManager:
    def __init__(self, hash_file_path: Path):
        self.hash_file_path = hash_file_path
        self.file_hashes = self._load()

    def _load(self) -> Dict[str, str]:
        if self.hash_file_path.exists():
            try:
                return json.load(open(self.hash_file_path, "r", encoding="utf-8"))
            except Exception as e:
                logger.warning(f"Could not load file hashes: {e}")
        return {}

    def save(self):
        try:
            json.dump(self.file_hashes, open(self.hash_file_path, "w", encoding="utf-8"),
                      ensure_ascii=False, indent=2)
        except Exception as e:
            logger.error(f"Could not save file hashes: {e}")

    def calculate(self, file_path: Path) -> str:
        sha = hashlib.sha256()
        try:
            with open(file_path, "rb") as f:
                for chunk in iter(lambda: f.read(4096), b""):
                    sha.update(chunk)
            return sha.hexdigest()
        except Exception as e:
            logger.error(f"Hash failed for {file_path}: {e}")
            return ""

    def changed(self, file_path: Path) -> bool:
        """Return True if file content changed since last save (keyed by absolute path)."""
        key = str(file_path.resolve())
        cur = self.calculate(file_path)
        if not cur:
            return True
        old = self.file_hashes.get(key)
        if old != cur:
            self.file_hashes[key] = cur
            return True
        return False

    def update(self, file_path: Path):
        """Store current hash (keyed by absolute path)."""
        key = str(file_path.resolve())
        h = self.calculate(file_path)
        if h:
            self.file_hashes[key] = h
            self.save()

class CheckpointManager:
    """
    Persist simple per-doc step markers so Stage 2 can resume mid-file.
    Stored at CHECKPOINT_PATH as: { "<stem>": ["semantic_saved","enriched_saved", ...] }
    """
    def __init__(self, path: Path):
        self.path = path
        self._data = self._load()

    def _load(self) -> Dict[str, List[str]]:
        try:
            with open(self.path, "r", encoding="utf-8") as f:
                obj = json.load(f)
                return obj if isinstance(obj, dict) else {}
        except Exception:
            return {}

    def _save(self):
        self.path.parent.mkdir(parents=True, exist_ok=True)
        tmp = self.path.with_suffix(".tmp")
        with open(tmp, "w", encoding="utf-8") as f:
            json.dump(self._data, f, ensure_ascii=False, indent=2)
        os.replace(tmp, self.path)

    def done(self, stem: str, step: str) -> bool:
        return step in self._data.get(stem, [])

    def mark(self, stem: str, step: str):
        steps = self._data.setdefault(stem, [])
        if step not in steps:
            steps.append(step)
            self._save()

    def clear(self, stem: str):
        if stem in self._data:
            del self._data[stem]
            self._save()

# ------------------
# Stage 1: Extraction (parallel, robust)
# ------------------

class TextExtractor:
    """Robust text+image extraction with PyMuPDF primary and PyPDF2 fallback."""
    def extract(self, pdf_path: Path) -> Optional[Dict]:
        try:
            doc = self._open_pdf_robust(pdf_path)
            if doc is None:
                return self._pypdf2_fallback(pdf_path)

            data = {
                "filename": pdf_path.name,
                "total_pages": len(doc),
                "text_chunks": [],
                "images": [],
                "tables": [],
                "metadata": {},
            }

            images_dir = RAG_OUTPUT_DIR / "images"
            images_dir.mkdir(parents=True, exist_ok=True)

            for pno in range(len(doc)):
                try:
                    page = doc[pno]
                    txt = page.get_text()
                    if txt and txt.strip():
                        data["text_chunks"].append({
                            "chunk_id": f"{pdf_path.stem}_page_{pno+1}",
                            "page_number": pno + 1,
                            "text": txt.strip(),
                            "word_count": len(txt.split()),
                            "char_count": len(txt),
                        })

                    # images: try pixmap then raw
                    try:
                        imgs = page.get_images(full=True)
                        for idx, img in enumerate(imgs):
                            try:
                                xref = img[0]
                                saved = False
                                # Pixmap path
                                try:
                                    pix = fitz.Pixmap(doc, xref)
                                    if pix.n - pix.alpha >= 4:
                                        pix = fitz.Pixmap(fitz.csRGB, pix)
                                    img_fn = f"{pdf_path.stem}_p{pno+1}_img{idx+1}.png"
                                    img_path = images_dir / img_fn
                                    pix.save(str(img_path))
                                    pix = None
                                    saved = True
                                    data["images"].append({
                                        "image_id": f"{pdf_path.stem}_p{pno+1}_img{idx+1}",
                                        "page_number": pno+1,
                                        "filename": img_fn,
                                        "file_path": str(img_path),
                                    })
                                except Exception:
                                    pass

                                # Raw path
                                if not saved:
                                    try:
                                        base = doc.extract_image(xref)
                                        raw = base["image"]
                                        ext = base.get("ext", "png")
                                        img_fn = f"{pdf_path.stem}_p{pno+1}_img{idx+1}_raw.{ext}"
                                        img_path = images_dir / img_fn
                                        with open(img_path, "wb") as out:
                                            out.write(raw)
                                        data["images"].append({
                                            "image_id": f"{pdf_path.stem}_p{pno+1}_img{idx+1}_raw",
                                            "page_number": pno+1,
                                            "filename": img_fn,
                                            "file_path": str(img_path),
                                        })
                                    except Exception:
                                        pass
                            except Exception as e:
                                logger.warning(f"Image extract failed p{pno+1} idx{idx+1}: {e}")
                    except Exception as e:
                        logger.warning(f"get_images failed on p{pno+1}: {e}")

                except Exception as e:
                    logger.warning(f"Page {pno+1} processing error: {e}")

            doc.close()
            return data

        except Exception as e:
            logger.error(f"Extraction fatal for {pdf_path.name}: {e}", exc_info=True)
            return None

    def _open_pdf_robust(self, pdf_path: Path):
        try:
            return fitz.open(pdf_path)
        except Exception as e:
            s = str(e).lower()
            if "graphics state" in s:
                try:
                    return fitz.open(str(pdf_path))
                except Exception:
                    try:
                        with tempfile.NamedTemporaryFile(suffix=".pdf", delete=False) as tmp:
                            tmp.write(open(pdf_path, "rb").read())
                            tmp.flush()
                            d = fitz.open(tmp.name)
                            os.unlink(tmp.name)
                            return d
                    except Exception:
                        return None
            return None

    def _pypdf2_fallback(self, pdf_path: Path) -> Optional[Dict]:
        try:
            data = {
                "filename": pdf_path.name,
                "total_pages": 0,
                "text_chunks": [],
                "images": [],
                "tables": [],
                "metadata": {"extraction_method": "pypdf2_fallback"},
            }
            with open(pdf_path, "rb") as f:
                reader = PyPDF2.PdfReader(f)
                data["total_pages"] = len(reader.pages)
                for pno, page in enumerate(reader.pages):
                    try:
                        txt = page.extract_text() or ""
                        if txt.strip():
                            data["text_chunks"].append({
                                "chunk_id": f"{pdf_path.stem}_page_{pno+1}",
                                "page_number": pno+1,
                                "text": txt.strip(),
                                "word_count": len(txt.split()),
                                "char_count": len(txt),
                            })
                    except Exception:
                        continue
            # We can attempt a light image pass even here
            try:
                d = fitz.open(str(pdf_path))
                imgs_dir = RAG_OUTPUT_DIR / "images"; imgs_dir.mkdir(parents=True, exist_ok=True)
                for pno in range(min(len(d), data["total_pages"])):
                    page = d[pno]
                    for idx, img in enumerate(page.get_images() or []):
                        try:
                            base = d.extract_image(img[0])
                            raw, ext = base["image"], base.get("ext", "png")
                            fn = f"{pdf_path.stem}_p{pno+1}_img{idx+1}_alt.{ext}"
                            fp = imgs_dir / fn
                            with open(fp, "wb") as out:
                                out.write(raw)
                            data["images"].append({
                                "image_id": f"{pdf_path.stem}_p{pno+1}_img{idx+1}_alt",
                                "page_number": pno+1,
                                "filename": fn,
                                "file_path": str(fp),
                            })
                        except Exception:
                            continue
                d.close()
            except Exception:
                pass
            return data
        except Exception as e:
            logger.error(f"PyPDF2 fallback failed for {pdf_path.name}: {e}")
            return None

# ------------------
# Prefilter (fixed search_for: no hit_max)
# ------------------

class PrefilterHelper:
    def __init__(self,
                 ctx_pages: int = PREFILTER_CONTEXT_PAGES,
                 max_hits: int = PREFILTER_MAX_HITS,
                 keywords: List[str] = PREFILTER_KEYWORDS):
        self.ctx_pages = ctx_pages
        self.max_hits  = max_hits
        self.keywords  = [k.lower() for k in keywords]

    def maybe_prefilter_huge(self, pdf_path: Path) -> Path:
        size = pdf_path.stat().st_size if pdf_path.exists() else 0
        try:
            doc = fitz.open(pdf_path)
            n_pages = len(doc)
            doc.close()
        except Exception:
            n_pages = 0
        if (n_pages >= HUGE_PDF_PAGES_THRESHOLD) or (size >= HUGE_PDF_BYTES_THRESHOLD):
            logger.info(f"Prefiltering huge PDF ({n_pages} pages, {size/1e6:.1f} MB): {pdf_path.name}")
            p, kept = self._prefilter_by_keywords(pdf_path)
            if kept > 0:
                logger.info(f"Prefilter kept {kept} pages → {p.name}")
                return p
        return pdf_path

    def _prefilter_by_keywords(self, pdf_path: Path) -> Tuple[Path, int]:
        doc = fitz.open(pdf_path)
        try:
            ranges = []
            # TOC-based
            try:
                toc = doc.get_toc(simple=True)
            except Exception:
                toc = []

            def toc_cut():
                if not toc: return False
                for i, (lvl, title, page) in enumerate(toc):
                    t = (title or "").lower()
                    if any(k in t for k in self.keywords):
                        start = max(page - 1 - self.ctx_pages, 0)
                        end = doc.page_count - 1
                        for j in range(i+1, len(toc)):
                            l2, t2, p2 = toc[j]
                            if l2 <= lvl:
                                end = max(p2 - 2 + self.ctx_pages, start)
                                break
                        ranges.append((start, min(end, doc.page_count - 1)))
                        return True
                return False

            found_toc = toc_cut()

            hits = 0
            if not found_toc:
                for pno in range(doc.page_count):
                    page = doc[pno]
                    found = False
                    for kw in self.keywords:
                        try:
                            rects = page.search_for(kw)  # version-safe (no hit_max)
                            if rects:
                                found = True
                                break
                        except Exception:
                            # text fallback if search_for barks
                            text = (page.get_text("text") or "").lower()
                            if kw in text:
                                found = True
                                break
                    if found:
                        s = max(0, pno - self.ctx_pages)
                        e = min(doc.page_count - 1, pno + self.ctx_pages)
                        ranges.append((s, e))
                        hits += 1
                        if hits >= self.max_hits:
                            break

            if not ranges:
                return pdf_path, 0

            ranges.sort()
            merged = []
            for s, e in ranges:
                if not merged or s > merged[-1][1] + 1:
                    merged.append([s, e])
                else:
                    merged[-1][1] = max(merged[-1][1], e)

            sub = fitz.open()
            kept = 0
            for s, e in merged:
                sub.insert_pdf(doc, from_page=s, to_page=e)
                kept += (e - s + 1)
            out_path = pdf_path.with_name(pdf_path.stem + "_leish_filter.pdf")
            sub.save(out_path, deflate=True, garbage=4)
            sub.close()
            return out_path, kept
        finally:
            doc.close()

# ------------------
# Stage 2: Heading extraction + semantic chunking + enrichment
# ------------------

class HeadingExtractor:
    def __init__(self):
        self.heading_patterns = [
            r'^(INTRODUCTION|CASE\s+PRESENTATION|DIAGNOSIS|TREATMENT|DISCUSSION|CONCLUSION|ABSTRACT|SUMMARY|BACKGROUND|METHODS|RESULTS)',
            r'^[0-9]+\.\s+[A-Z][A-Za-z\s]+',
            r'^[A-Z][A-Z\s]{5,}$',
        ]

    def _simple_text(self, pdf_path: Path) -> Optional[Dict]:
        try:
            doc = fitz.open(pdf_path)
            texts = []
            for page in doc:
                texts.append(page.get_text("text"))
            doc.close()
            full = " ".join([t for t in texts if t])
            return {
                "filename": pdf_path.name,
                "sections": [{
                    "heading": "Full Document",
                    "level": 1,
                    "content": full,
                    "page_start": 1,
                    "page_end": len(texts)
                }],
                "full_text": full,
                "headings": [{"text": "Full Document", "level": 1, "page": 1}],
                "fonts": {}
            }
        except Exception as e:
            logger.error(f"Simple text extraction failed for {pdf_path.name}: {e}")
            return None

    def extract(self, pdf_path: Path) -> Optional[Dict]:
        try:
            doc = fitz.open(pdf_path)
            structure = {
                "filename": pdf_path.name,
                "sections": [],
                "full_text": "",
                "headings": [],
                "fonts": {}
            }
            current = {
                "heading": "Introduction",
                "level": 1,
                "content": "",
                "page_start": 1,
                "page_end": 1
            }

            for pno, page in enumerate(doc):
                blocks = page.get_text("dict")
                for block in blocks.get("blocks", []):
                    if "lines" not in block: continue
                    for line in block["lines"]:
                        line_text = ""
                        line_fonts = []
                        for span in line["spans"]:
                            text = span["text"].strip()
                            if not text: continue
                            line_text += text + " "
                            line_fonts.append({
                                "size": span["size"],
                                "font": span["font"],
                                "flags": span["flags"],
                            })
                        line_text = line_text.strip()
                        if not line_text: continue
                        is_h, lvl = self._is_heading(line_text, line_fonts)
                        if is_h:
                            if current["content"].strip():
                                current["page_end"] = pno + 1
                                structure["sections"].append(current.copy())
                            current = {
                                "heading": line_text,
                                "level": lvl,
                                "content": "",
                                "page_start": pno+1,
                                "page_end": pno+1
                            }
                            structure["headings"].append({
                                "text": line_text, "level": lvl, "page": pno+1
                            })
                        else:
                            current["content"] += line_text + " "
                        structure["full_text"] += line_text + " "
            if current["content"].strip():
                current["page_end"] = len(doc)
                structure["sections"].append(current)
            doc.close()

            if len(structure["full_text"].strip()) < 80 or len(structure["sections"]) == 0:
                logger.warning("Structured extraction thin. Falling back to simple.")
                simple = self._simple_text(pdf_path)
                if simple and len(simple["full_text"].strip()) >= 40:
                    return simple
            return structure
        except Exception as e:
            logger.error(f"Heading extraction error for {pdf_path.name}: {e}")
            return self._simple_text(pdf_path)

    def _is_heading(self, text: str, fonts: List[Dict]) -> Tuple[bool, int]:
        if not text or len(text.strip()) < 3: return (False, 0)
        for pat in self.heading_patterns:
            if re.match(pat, text.upper().strip()):
                return (True, self._level(text))
        if fonts:
            avg = sum(f["size"] for f in fonts) / len(fonts)
            is_bold = any(f["flags"] & 2**4 for f in fonts)  # bold flag
            if (avg > 12 or is_bold) and len(text) < 100:
                return (True, self._level(text))
        return (False, 0)

    def _level(self, text: str) -> int:
        t = text.upper().strip()
        if any(s in t for s in ["INTRODUCTION", "ABSTRACT", "CONCLUSION", "DISCUSSION"]):
            return 1
        if re.match(r"^[0-9]+\.", t): return 2
        if re.match(r"^[0-9]+\.[0-9]+", t): return 3
        return 2

def render_pdf_pages_to_png(pdf_path: Path, out_root: Path = PAGE_RENDERS_DIR, dpi: int = 150) -> List[Dict]:
    """
    Render each PDF page to PNG for ColQwen2. Idempotent: skips if already present.
    Returns list of dict rows with doc_id/page/image_path/width/height/source_file/row_id.
    """
    try:
        doc_id = pdf_path.stem
        out_dir = out_root / doc_id
        out_dir.mkdir(parents=True, exist_ok=True)
        meta = []
        doc = fitz.open(pdf_path)
        try:
            for i in range(len(doc)):
                try:
                    page = doc[i]
                    page_no = i + 1
                    img_name = f"page_{page_no:04d}.png"
                    img_path = out_dir / img_name
                    if not img_path.exists():
                        # render at requested dpi
                        zoom = dpi / 72.0
                        mat = fitz.Matrix(zoom, zoom)
                        pix = page.get_pixmap(matrix=mat, alpha=False)
                        pix.save(str(img_path))
                except Exception as e:
                    logger.warning(f"Failed to render page {i+1} of {pdf_path.name}: {e}")
                    continue
                # widths/heights are in pixels at requested dpi
                meta.append({
                    "source_file": str(pdf_path),
                    "doc_id": doc_id,
                    "page": page_no,
                    "image_path": str(img_path),
                    "width": page.rect.width * (dpi/72.0),
                    "height": page.rect.height * (dpi/72.0),
                    "row_id": len(meta)
                })
        finally:
            doc.close()
        # write per-doc metadata json (what Code B loads first)
        META_DIR.mkdir(parents=True, exist_ok=True)
        with open(META_DIR / f"{doc_id}_images.json", "w", encoding="utf-8") as f:
            json.dump(meta, f, ensure_ascii=False, indent=2)
        return meta
    except Exception as e:
        logger.error(f"Page rendering failed for {pdf_path.name}: {e}")
        return []

class SemanticChunker:
    def __init__(self):
        try:
            logger.info("Loading sentence embedding model (offline preferred)...")
            self.embedding_model = load_sentence_embedder(EMBEDDING_MODEL_NAME)
        except Exception as e:
            logger.warning(f"Embedding model failed to load: {e}")
            self.embedding_model = None
        self.nlp = ensure_spacy_model()

        self.section_markers = {
            'case_presentation': ['case presentation', 'patient', 'admitted', 'presented with'],
            'diagnosis': ['diagnosis', 'diagnostic', 'differential diagnosis', 'confirmed by'],
            'treatment': ['treatment', 'therapy', 'medication', 'administered', 'prescribed'],
            'discussion': ['discussion', 'in conclusion', 'this case illustrates'],
            'methodology': ['methods', 'procedure', 'protocol', 'conducted'],
            'results': ['results', 'findings', 'observed', 'demonstrated']
        }

    

    def create_chunks(self, doc_struct: Dict) -> List[Dict]:
        all_chunks = []
        for sidx, section in enumerate(doc_struct["sections"]):
            cs = self._chunk_section(section, doc_struct["filename"], sidx)
            all_chunks.extend(cs)
        return all_chunks

    def _chunk_section(self, section: Dict, filename: str, sidx: int) -> List[Dict]:
        content = section["content"].strip()
        if len(content) < MIN_CHUNK_SIZE:
            return [self._mk_chunk(content, section, filename, sidx, 0, "tiny_fallback")]
        sents = self._segment(content)
        if self.embedding_model and len(sents) >= 2:
            try:
                emb = self.embedding_model.encode(sents, batch_size=64, show_progress_bar=not DISABLE_PROGRESS_BAR)
                total_chars = sum(len(s) for s in sents)
                k = max(1, total_chars // MAX_CHUNK_SIZE)
                k = min(k, len(sents))
                if k == 1:
                    clusters = [0]*len(sents)
                else:
                    try:
                        clustering = AgglomerativeClustering(n_clusters=k, linkage="average", metric="cosine")
                    except TypeError:
                        clustering = AgglomerativeClustering(n_clusters=k, linkage="average", affinity="cosine")
                    clusters = clustering.fit_predict(emb)
                groups = {}
                for i, cid in enumerate(clusters):
                    groups.setdefault(int(cid), []).append((i, sents[i]))
                chunks = []
                for cid, group in groups.items():
                    group.sort(key=lambda x: x[0])
                    text = " ".join(s for _, s in group)
                    if len(text) < MIN_CHUNK_SIZE:
                        continue
                    if len(text) > MAX_CHUNK_SIZE:
                        for sub in self._split(text):
                            chunks.append(self._mk_chunk(sub, section, filename, sidx, len(chunks), "embedding_split"))
                    else:
                        chunks.append(self._mk_chunk(text, section, filename, sidx, len(chunks), "embedding_cluster"))
                if chunks:
                    return chunks
            except Exception as e:
                logger.warning(f"Embedding chunking failed; falling back. Reason: {e}")

        # rule-based
        return self._rule_chunk(sents, section, filename, sidx)

    def _segment(self, text: str) -> List[str]:
        if self.nlp:
            doc = self.nlp(text)
            return [s.text.strip() for s in doc.sents if len(s.text.strip()) > 8]
        return [s.strip() for s in sent_tokenize(text) if len(s.strip()) > 8]

    def _rule_chunk(self, sents: List[str], section: Dict, filename: str, sidx: int) -> List[Dict]:
        chunks, cur, cur_type = [], "", None
        for s in sents:
            st = self._disc_type(s)
            if (cur_type and st != cur_type) or (len(cur) + len(s) > MAX_CHUNK_SIZE):
                if len(cur.strip()) >= MIN_CHUNK_SIZE:
                    chunks.append(self._mk_chunk(cur.strip(), section, filename, sidx, len(chunks), "rule_based", cur_type))
                cur, cur_type = s + " ", st
            else:
                cur += s + " "
                if not cur_type: cur_type = st
        if len(cur.strip()) >= MIN_CHUNK_SIZE:
            chunks.append(self._mk_chunk(cur.strip(), section, filename, sidx, len(chunks), "rule_based", cur_type))
        return chunks

    def _disc_type(self, s: str) -> str:
        sl = s.lower()
        for t, markers in self.section_markers.items():
            if any(m in sl for m in markers): return t
        return "general_medical"

    def _split(self, text: str) -> List[str]:
        sents = self._segment(text)
        out, cur = [], ""
        for s in sents:
            if len(cur) + len(s) > MAX_CHUNK_SIZE and cur:
                out.append(cur.strip()); cur = s + " "
            else:
                cur += s + " "
        if cur.strip(): out.append(cur.strip())
        return out

    def _mk_chunk(self, content: str, section: Dict, filename: str, sidx: int, cidx: int, method: str, disc: str = None) -> Dict:
        cid = f"{Path(filename).stem}_s{sidx}_c{cidx}"
        return {
            "chunk_id": cid,
            "source_file": filename,
            "section_heading": section["heading"],
            "section_level": section["level"],
            "content": content,
            "char_count": len(content),
            "word_count": len(content.split()),
            "chunking_method": method,
            "discourse_type": disc or "unknown",
            "section_pages": f"{section['page_start']}-{section['page_end']}",
            "created_at": datetime.now().isoformat(),
            "hash": hashlib.md5(content.encode()).hexdigest()[:12],
        }

class MixtralEnricher:
    """Deterministic enrichment (do_sample=False) with 4-bit quantization if possible."""
    def __init__(self):
        try:
            logger.info(f"Loading LLM (offline preferred): {LLM_MODEL_NAME}")
            self.pipe = load_llm(LLM_MODEL_NAME)
        except Exception as e:
            logger.error(f"Failed to load LLM locally: {e}")
            self.pipe = None

        self.examples = [
            {
                "input": "Leishmaniasis is a vector-borne disease caused by protozoan parasites of the genus Leishmania and transmitted by phlebotomine sandflies. The disease is endemic in tropical and subtropical regions worldwide.",
                "output": {
                    "summary": "Leishmaniasis is a sandfly-transmitted parasitic disease endemic to tropical regions.",
                    "medical_keywords": ["leishmaniasis", "vector-borne disease", "Leishmania", "protozoan parasites", "phlebotomine sandflies", "endemic", "tropical", "subtropical"],
                    "content_type": "disease_definition",
                    "relevance": 1.0,
                    "clinical_significance": "high"
                }
            }
        ]

    def enrich_batch(self, chunks: List[Dict]) -> List[Dict]:
        if not self.pipe:
            logger.warning("LLM not available; returning empty enrichment.")
            return []
        kept, buffer = [], []
        for start in range(0, len(chunks), BATCH_SIZE_GENERATION):
            batch = chunks[start:start+BATCH_SIZE_GENERATION]
            prompts = [self._make_prompt(c["content"]) for c in batch]
            try:
                out = self.pipe(
                    prompts,
                    max_new_tokens=400,
                    do_sample=False,
                    return_full_text=False,
                    pad_token_id=self.pipe.tokenizer.eos_token_id,
                    max_time=MAX_ENRICH_SECONDS_PER_CHUNK
                )
            except Exception as e:
                logger.error(f"Batch generation failed: {e}")
                out = [None]*len(batch)

            for c, resp in zip(batch, out):
                enrich = self._parse(resp)
                c.update({
                    "llm_summary": enrich.get("summary", c["content"][:100] + "..."),
                    "medical_keywords": enrich.get("medical_keywords", []),
                    "llm_content_type": enrich.get("content_type", "unknown"),
                    "leishmania_relevance": float(enrich.get("relevance", 0.0)),
                    "clinical_significance": enrich.get("clinical_significance", "unknown"),
                    "enrichment_timestamp": datetime.now().isoformat(),
                })
                if c["leishmania_relevance"] >= MIN_RELEVANCE_SCORE:
                    buffer.append(c)
        kept.extend(buffer)
        return kept

    def _make_prompt(self, content: str) -> str:
        ex = self.examples[0]
        return f"""[INST] You are a medical AI specializing in Leishmaniasis.
Given the text, return ONLY JSON with fields:
summary (1 sentence), medical_keywords (list), content_type (one of disease_definition, pathophysiology, clinical_presentation, diagnosis, treatment, epidemiology, case_report, general_medical, reference), relevance (0-1), clinical_significance (high/medium/low).

Example:
Input: "{ex['input']}"
Output: {json.dumps(ex['output'], ensure_ascii=False)}

Now analyze:
Input: "{content}" [/INST]"""

    def _parse(self, resp: Any) -> Dict:
        try:
            if resp is None: return {}
            txt = resp[0]["generated_text"] if isinstance(resp, list) else resp["generated_text"]
            s = txt.find("{"); e = txt.rfind("}")
            if s != -1 and e != -1:
                payload = txt[s:e+1]
                obj = json.loads(payload)
                # sanitize
                obj["summary"] = str(obj.get("summary", ""))[:200]
                kws = obj.get("medical_keywords", [])
                if not isinstance(kws, list): kws = []
                obj["medical_keywords"] = [str(k).lower().strip() for k in kws[:20] if len(str(k).strip())>2]
                if obj.get("content_type") not in {"disease_definition","pathophysiology","clinical_presentation","diagnosis","treatment","epidemiology","case_report","general_medical","reference"}:
                    obj["content_type"] = "general_medical"
                try:
                    obj["relevance"] = max(0.0, min(1.0, float(obj.get("relevance", 0.0))))
                except Exception:
                    obj["relevance"] = 0.0
                if obj.get("clinical_significance") not in {"high","medium","low"}:
                    obj["clinical_significance"] = "unknown"
                return obj
        except Exception:
            pass
        return {}

# ------------------
# Writers / Savers
# ------------------

class RAGWriter:
    def __init__(self):
        (RAG_OUTPUT_DIR / "chunks").mkdir(parents=True, exist_ok=True)
        (RAG_OUTPUT_DIR / "images").mkdir(parents=True, exist_ok=True)
        (RAG_OUTPUT_DIR / "metadata").mkdir(parents=True, exist_ok=True)
        (PAGE_RENDERS_DIR).mkdir(parents=True, exist_ok=True)
        (META_DIR).mkdir(parents=True, exist_ok=True)
        (EVIDENCE_UNITS_DIR).mkdir(parents=True, exist_ok=True)
        (FINETUNE_OUTPUT_DIR / "qa_pairs").mkdir(parents=True, exist_ok=True)
        (FINETUNE_OUTPUT_DIR / "summaries").mkdir(parents=True, exist_ok=True)
        (Path("kaggle/working_v2/enriched_chunks/core")).mkdir(parents=True, exist_ok=True)
        (Path("kaggle/working_v2/enriched_chunks/longtail")).mkdir(parents=True, exist_ok=True)

    def save_semantic(self, chunks: List[Dict], stem: str):
        out = Path("kaggle/working_v2/semantic_chunks") / f"{stem}_semantic_chunks.json"
        out.parent.mkdir(parents=True, exist_ok=True)
        json.dump(chunks, open(out, "w", encoding="utf-8"), ensure_ascii=False, indent=2)

    def save_enriched(self, core: List[Dict], longtail: List[Dict], stem: str):
        core_p = Path("kaggle/working_v2/enriched_chunks/core") / f"{stem}_core_chunks.json"
        lt_p   = Path("kaggle/working_v2/enriched_chunks/longtail") / f"{stem}_longtail_chunks.json"
        all_p  = RAG_OUTPUT_DIR / "chunks" / f"{stem}_chunks.json"
        json.dump(core, open(core_p, "w", encoding="utf-8"), ensure_ascii=False, indent=2)
        if longtail:
            json.dump(longtail, open(lt_p, "w", encoding="utf-8"), ensure_ascii=False, indent=2)
        json.dump(core+longtail, open(all_p, "w", encoding="utf-8"), ensure_ascii=False, indent=2)

    # --- Q&A and Summaries (from Code A spirit; compact) ---

    def _classify_q(self, q: str) -> str:
        ql = q.lower()
        if ql.startswith("what"): return "definition"
        if ql.startswith("how"):  return "process"
        if ql.startswith("when"): return "temporal"
        if ql.startswith("where"):return "location"
        if ql.startswith("why"):  return "causal"
        return "general"

    def _extract_answer(self, question: str, content: str, keywords: List[str]) -> str:
        sentences = sent_tokenize(content)
        q_words = set(word_tokenize(question.lower())) - set(stopwords.words("english"))
        scored = []
        for s in sentences:
            sl = s.lower()
            s_words = set(word_tokenize(sl))
            overlap = len(q_words & s_words) / (len(q_words) or 1)
            kw_score = sum(1 for kw in keywords if kw in sl) * 0.3
            med_bonus = sum(0.2 for t in ["treatment","diagnosis","patient","clinical","therapy","disease","condition"] if t in sl)
            total = overlap + kw_score + med_bonus
            if total > 0 and len(s.strip()) > 20:
                scored.append((s.strip(), total))
        scored.sort(key=lambda x: x[1], reverse=True)
        top = [s for s,_ in scored[:3]]
        ans = ". ".join(top) if top else " ".join(sentences[:2])[:300]
        return ans

    def generate_qa(self, enriched: List[Dict], doc_struct: Dict) -> List[Dict]:
        qa = []
        by_type = {}
        for c in enriched:
            by_type.setdefault(c.get("llm_content_type","unknown"), []).append(c)

        templates = {
            'disease_definition': [
                "What disease is being described?",
                "How is this condition defined?",
                "What are the key characteristics mentioned?"
            ],
            'pathophysiology': [
                "What is the underlying mechanism described?",
                "How does this biological process work?",
                "What biological factors are involved?"
            ],
            'clinical_presentation': [
                "What are the clinical signs and symptoms?",
                "How does the patient typically present?",
                "What clinical features are described?"
            ],
            'diagnosis': [
                "What diagnostic methods are mentioned?",
                "How is this condition diagnosed?",
                "What tests are recommended?"
            ],
            'treatment': [
                "What treatment options are described?",
                "What is the recommended therapy?",
                "What medications are mentioned?"
            ],
            'epidemiology': [
                "What is the epidemiological pattern described?",
                "What populations are affected?",
                "What is the geographic distribution?"
            ],
            'case_report': [
                "What was the patient's presentation?",
                "What was the diagnosis and treatment?",
                "What can we learn from this case?"
            ]
        }

        for ctype, items in by_type.items():
            qs = templates.get(ctype, [
                "What is the main point discussed?",
                "What medical information is provided?",
                "What key concepts are explained?"
            ])
            for c in items:
                for i, q in enumerate(qs[:3]):
                    qa.append({
                        "question_id": f"{c['chunk_id']}_qa_{i}",
                        "source_file": c["source_file"],
                        "chunk_id": c["chunk_id"],
                        "question": q,
                        "answer": self._extract_answer(q, c["content"], c.get("medical_keywords", [])),
                        "context": c["content"][:500],
                        "content_type": ctype,
                        "question_type": self._classify_q(q),
                        "relevance_score": c.get("leishmania_relevance", 0.0),
                        "keywords": c.get("medical_keywords", []),
                        "created_at": datetime.now().isoformat(),
                        "generation_method": "template"
                    })
        # small cross-link
        diag = [c for c in enriched if c.get("llm_content_type")=="diagnosis"]
        treat= [c for c in enriched if c.get("llm_content_type")=="treatment"]
        if diag and treat:
            ctx = (diag[0]["content"] + " " + treat[0]["content"])[:1000]
            qa.append({
                "question_id": f"{doc_struct['filename']}_cross_diag_treat",
                "source_file": doc_struct["filename"],
                "chunk_id": f"{diag[0]['chunk_id']},{treat[0]['chunk_id']}",
                "question": "What is the relationship between diagnosis and treatment described?",
                "answer": self._extract_answer("How are diagnosis and treatment connected?", ctx, []),
                "context": ctx,
                "content_type": "cross_reference",
                "question_type": "relationship",
                "generation_method": "cross_chunk",
                "created_at": datetime.now().isoformat()
            })
        return qa

    def save_qa(self, qa: List[Dict], stem: str):
        out = FINETUNE_OUTPUT_DIR / "qa_pairs" / f"{stem}_enhanced_qa.json"
        json.dump(qa, open(out, "w", encoding="utf-8"), ensure_ascii=False, indent=2)

        # training format
        training = []
        for q in qa:
            training.append({
                "instruction": q["question"],
                "input": q.get("context","")[:500],
                "output": q["answer"],
                "metadata": {
                    "source": q["source_file"],
                    "content_type": q.get("content_type","unknown"),
                    "relevance_score": q.get("relevance_score",0.0),
                    "question_type": q.get("question_type","general"),
                }
            })
        out2 = FINETUNE_OUTPUT_DIR / "qa_pairs" / f"{stem}_training_format.json"
        json.dump(training, open(out2, "w", encoding="utf-8"), ensure_ascii=False, indent=2)

    def save_summary(self, doc_struct: Dict, core: List[Dict], longtail: List[Dict], stem: str):
        allc = core + longtail
        kw_freq = {}
        for c in allc:
            for k in c.get("medical_keywords", []):
                kw_freq[k] = kw_freq.get(k, 0) + 1
        top_kw = sorted(kw_freq.items(), key=lambda x: x[1], reverse=True)[:20]
        dist = {}
        for c in allc:
            t = c.get("llm_content_type","unknown")
            dist[t] = dist.get(t,0)+1
        rels = [c.get("leishmania_relevance",0.0) for c in allc]
        avg_rel = sum(rels)/len(rels) if rels else 0.0
        summary = {
            "document_id": stem,
            "source_file": doc_struct["filename"],
            "processing_method": "merged_semantic_llm",
            "created_at": datetime.now().isoformat(),
            "document_structure": {
                "total_sections": len(doc_struct["sections"]),
                "section_headings": [s["heading"] for s in doc_struct["sections"]],
                "total_text_length": len(doc_struct["full_text"]),
                "heading_hierarchy": doc_struct["headings"]
            },
            "chunk_statistics": {
                "total_chunks": len(allc),
                "core_chunks": len(core),
                "longtail_chunks": len(longtail),
                "avg_relevance_score": avg_rel,
                "content_type_distribution": dist,
                "avg_chunk_length": (sum(c["char_count"] for c in allc)/len(allc)) if allc else 0
            },
            "content_analysis": {
                "top_keywords": [k for k,_ in top_kw[:10]],
                "keyword_frequencies": dict(top_kw),
                "leishmania_relevance_distribution": {
                    "high (0.8+)": len([c for c in allc if c.get("leishmania_relevance",0)>=0.8]),
                    "medium (0.5-0.8)": len([c for c in allc if 0.5<=c.get("leishmania_relevance",0)<0.8]),
                    "low (0.3-0.5)": len([c for c in allc if 0.3<=c.get("leishmania_relevance",0)<0.5]),
                }
            },
            "summaries": {
                "executive": self._exec_summary(core),
                "detailed": self._detailed(allc, doc_struct),
                "core_insights": self._core_insights(core),
            }
        }
        out = FINETUNE_OUTPUT_DIR / "summaries" / f"{stem}_enhanced_summary.json"
        json.dump(summary, open(out, "w", encoding="utf-8"), ensure_ascii=False, indent=2)

    def save_evidence_units(self, doc_struct: Dict, enriched_chunks: List[Dict]):
        """
        For each rendered page image, attach a caption/snippet from the most relevant enriched chunk
        whose section_pages cover that page. Output one JSON per doc in EVIDENCE_UNITS_DIR.
        """
        try:
            doc_id = Path(doc_struct["filename"]).stem
            img_dir = PAGE_RENDERS_DIR / doc_id
            if not img_dir.exists():
                logger.info(f"No page renders for {doc_id}; skipping evidence_units.")
                return

            # Fast index: section page ranges -> chunks
            ranges = []
            for c in enriched_chunks:
                rng = c.get("section_pages", "1-1")
                try:
                    s, e = [int(x) for x in rng.split("-")]
                except Exception:
                    s, e = 1, 1
                ranges.append((s, e, c))

            evidence = []
            pages = sorted(img_dir.glob("page_*.png"))
            for pth in pages:
                m = re.search(r"page_(\d+)\.png", pth.name)
                page_no = int(m.group(1)) if m else 1
                best = None
                # choose first covering chunk; could be improved with heuristics
                for s, e, c in ranges:
                    if s <= page_no <= e:
                        best = c
                        break
                caption = ""
                tags = []
                if best:
                    caption = best.get("llm_summary") or ""
                    if not caption:
                        # fallback: first sentence of chunk content
                        sents = sent_tokenize(best.get("content", ""))
                        caption = sents[0] if sents else ""
                    tags = best.get("medical_keywords", [])[:12]
                evidence.append({
                    "id": f"{doc_id}_{page_no}_{pth.name}",
                    "doc_id": doc_id,
                    "page": page_no,
                    "figure_id": pth.name,
                    "modality": "image",
                    "image_path": str(pth),
                    "caption": caption,
                    "snippet": caption[:400],
                    "emb_scores": {},
                    "tags": tags,
                })

            if evidence:
                EVIDENCE_UNITS_DIR.mkdir(parents=True, exist_ok=True)
                with open(EVIDENCE_UNITS_DIR / f"{doc_id}.json", "w", encoding="utf-8") as f:
                    json.dump(evidence, f, ensure_ascii=False, indent=2)
                logger.info(f"Saved evidence_units for {doc_id}: {len(evidence)} pages.")
        except Exception as e:
            logger.warning(f"save_evidence_units failed: {e}")

    def _exec_summary(self, core: List[Dict]) -> str:
        if not core: return "No high-relevance content found."
        sents = []
        for c in core[:5]:
            sm = c.get("llm_summary","")
            if sm and len(sm)>20: sents.append(sm)
        if sents: return " ".join(sents)
        joined = " ".join([c["content"] for c in core[:3]])
        s = sent_tokenize(joined)
        picks = [x.strip() for x in s[:8] if any(t in x.lower() for t in ["leishmania","treatment","diagnosis","patient"])]
        return ". ".join(picks[:3]) or "Executive summary not available."

    def _detailed(self, allc: List[Dict], ds: Dict) -> str:
        by_sec: Dict[str, List[Dict]] = {}
        for c in allc: by_sec.setdefault(c.get("section_heading","Unknown"), []).append(c)
        parts = []
        for sec, items in by_sec.items():
            if not items: continue
            bits = []
            for c in items[:3]:
                if c.get("llm_summary"): bits.append(c["llm_summary"])
                else:
                    sents = sent_tokenize(c["content"])
                    if sents: bits.append(sents[0])
            if bits:
                parts.append(f"**{sec}**: " + " ".join(bits))
        return "\n\n".join(parts) if parts else "Detailed summary not available."

    def _core_insights(self, core: List[Dict]) -> List[str]:
        lab = {
            "disease_definition": "Disease Definition",
            "pathophysiology": "Pathophysiology",
            "clinical_presentation": "Clinical Presentation",
            "diagnosis": "Diagnostic Approach",
            "treatment": "Treatment Options",
            "epidemiology": "Epidemiological Context",
            "case_report": "Case Study Findings",
        }
        by = {}
        for c in core:
            t = c.get("llm_content_type","unknown")
            by.setdefault(t, []).append(c)
        out = []
        for t, items in by.items():
            label = lab.get(t, t.title().replace("_"," "))
            summ = [i.get("llm_summary","") for i in items if i.get("llm_summary")]
            if summ:
                out.append(f"{label}: {' '.join(summ[:2])}")
        return out[:8]

# ------------------
# Orchestrator
# ------------------

class MergedPipeline:
    def __init__(self):
        self.extractor  = TextExtractor()
        self.prefilter  = PrefilterHelper()
        self.heading    = HeadingExtractor()
        self.chunker    = SemanticChunker()
        self.enricher   = MixtralEnricher()
        self.writer     = RAGWriter()
        self.hash_mgr   = FileHashManager(FILE_HASHES_PATH)
        self.results: List[Dict] = []
        self.ckpt     = CheckpointManager(CHECKPOINT_PATH)

    def _save_metadata(self):
        try:
            df_new = pd.DataFrame(self.results)
            if PROCESSED_METADATA_PATH.exists():
                try:
                    df_prev = pd.read_csv(PROCESSED_METADATA_PATH)
                except Exception:
                    df_prev = pd.DataFrame()
                # Merge, keep latest row per filename
                df = pd.concat([df_prev, df_new], ignore_index=True)
                if "filename" in df.columns:
                    df = df.drop_duplicates(subset=["filename"], keep="last")
            else:
                df = df_new
            df.to_csv(PROCESSED_METADATA_PATH, index=False)
            logger.info(f"Saved processing metadata ({len(df)} records).")
        except Exception as e:
            logger.error(f"Could not save metadata: {e}")

    def _process_one_file(self, pdf_path: Path, process_id: int) -> Dict:
        try:
            start = time.time()
            logger.info(f"[{process_id}] Extracting: {pdf_path.name}")
            data = self.extractor.extract(pdf_path)
            if not data:
                return {"filename": pdf_path.name, "filepath": str(pdf_path), "status":"failed",
                        "error":"extraction_failed", "process_id": process_id, "processed_at": datetime.now().isoformat()}
            result = {
                "filename": pdf_path.name,
                "filepath": str(pdf_path),
                "processed_at": datetime.now().isoformat(),
                "total_pages": data.get("total_pages", 0),
                "text_chunks": len(data.get("text_chunks", [])),
                "images_extracted": len(data.get("images", [])),
                "tables_extracted": len(data.get("tables", [])),
                "status": "extracted",
                "process_id": process_id,
                "processing_time": time.time() - start
            }
            return result
        except Exception as e:
            return {"filename": pdf_path.name, "filepath": str(pdf_path), "status":"failed",
                    "error": str(e), "process_id": process_id, "processed_at": datetime.now().isoformat()}

    def stage1_parallel_extract(self, pdf_files: List[Path]) -> List[Dict]:
        infos = []
        for i, p in enumerate(pdf_files, 1):
            infos.append({"path": p, "id": i})
        results = []
        with ProcessPoolExecutor(max_workers=MAX_WORKERS) as ex:
            fut2file = { ex.submit(self._worker_wrapper, i["path"], i["id"]): i for i in infos }
            done = 0
            for fut in as_completed(fut2file):
                meta = fut2file[fut]
                done += 1
                try:
                    r = fut.result()
                    results.append(r)
                    logger.info(f"Progress: {done}/{len(infos)} — {r['filename']} [{r['status']}]")
                    if r["status"] != "failed":
                        self.hash_mgr.update(Path(r["filepath"]))
                except Exception as e:
                    logger.error(f"Worker failed for {meta['path'].name}: {e}")
                    results.append({"filename": meta["path"].name, "filepath": str(meta["path"]),
                                    "status":"failed","error":str(e), "process_id": meta["id"]})
        return results

    @staticmethod
    def _worker_wrapper(path: Path, pid: int) -> Dict:
        # Recreate lightweight extractor in subprocess
        extractor = TextExtractor()
        try:
            start = time.time()
            data = extractor.extract(path)
            if not data:
                return {"filename": path.name, "filepath": str(path), "status":"failed",
                        "error": "extraction_failed", "process_id": pid, "processed_at": datetime.now().isoformat()}
            return {
                "filename": path.name, "filepath": str(path),
                "processed_at": datetime.now().isoformat(),
                "total_pages": data.get("total_pages", 0),
                "text_chunks": len(data.get("text_chunks", [])),
                "images_extracted": len(data.get("images", [])),
                "tables_extracted": len(data.get("tables", [])),
                "status": "extracted", "process_id": pid,
                "processing_time": time.time() - start
            }
        except Exception as e:
            return {"filename": path.name, "filepath": str(path),
                    "status":"failed","error":str(e), "process_id": pid,
                    "processed_at": datetime.now().isoformat()}
        
    def _resolve_row_path(self, row: Dict) -> Optional[Path]:
        try:
            fp = row.get("filepath", "")
        except Exception:
            fp = ""
        p = Path(fp) if fp else (TEXTBOOK_SOURCE_DIR / row["filename"])
        if p.exists():
            return p
        # fallback: try source dir + filename
        cand = TEXTBOOK_SOURCE_DIR / row.get("filename", "")
        return cand if cand.exists() else None

    def _find_pdf_by_stem(self, stem: str) -> Optional[Path]:
        # 1) quick scan of source dir by name
        cand = list(TEXTBOOK_SOURCE_DIR.rglob(f"{stem}.pdf"))
        if cand:
            return cand[0]
        # 2) broader prefix match
        cand = list(TEXTBOOK_SOURCE_DIR.rglob(f"{stem}*.pdf"))
        if cand:
            return cand[0]
        return None

    def _collect_failed_or_checkpointed(self) -> List[Path]:
        paths: List[Path] = []
        seen: Set[str] = set()

        # From metadata: failed or extracted-only (crash between stages)
        if PROCESSED_METADATA_PATH.exists():
            try:
                df_prev = pd.read_csv(PROCESSED_METADATA_PATH)
                mask = df_prev["status"].isin(["failed", "extracted"])
                for _, row in df_prev[mask].iterrows():
                    p = self._resolve_row_path(row)
                    if p and p.exists() and p.name not in seen:
                        seen.add(p.name)
                        paths.append(p)
            except Exception as e:
                logger.warning(f"Could not read previous metadata for resume: {e}")

        # From checkpoints: anything with partial steps recorded
        try:
            # access the underlying checkpoint data
            stems = list(getattr(self.ckpt, "_data", {}).keys())
            for stem in stems:
                p = self._find_pdf_by_stem(stem)
                if p and p.exists() and p.name not in seen:
                    seen.add(p.name)
                    paths.append(p)
        except Exception as e:
            logger.warning(f"Could not read checkpoints for resume: {e}")

        return paths
    
    def run_failed_only(self):
        if not TEXTBOOK_SOURCE_DIR.exists():
            logger.error(f"Source dir not found: {TEXTBOOK_SOURCE_DIR}")
            return

        subset = self._collect_failed_or_checkpointed()
        if not subset:
            logger.info("No failed or checkpointed documents to resume. You're all caught up.")
            return

        logger.info(f"Will resume {len(subset)} file(s): " + ", ".join(p.name for p in subset[:8]) + ("..." if len(subset) > 8 else ""))

        # Re-run Stage 1 for the subset (idempotent; cheap vs. debugging conditional branches)
        stage1 = self.stage1_parallel_extract(subset)
        self.results.extend(stage1)
        self._save_metadata()

        # Stage 2 for successful extractions (or those marked 'extracted')
        successes = [r for r in stage1 if r.get("status") != "failed"]
        for i, r in enumerate(successes, 1):
            pdf_path = Path(r["filepath"])
            logger.info(f"[Stage2 resume {i}/{len(successes)}] {pdf_path.name}")
            res2 = self.stage2_enrich(pdf_path)

            # Merge/replace row in self.results (this turn)
            self.results = [x for x in self.results if x["filename"] != r["filename"]]
            merged = {**r, **res2}
            self.results.append(merged)

            # Persist back into the global CSV (merge with previous)
            self._save_metadata()

        self._print_final()

    def stage2_enrich(self, pdf_path: Path) -> Dict:
        """Stage 2 with mid-file resume via CheckpointManager."""
        stem = pdf_path.stem
        try:
            start = time.time()

            # Prefilter only for huge PDFs (pure read; no checkpointing needed)
            pdf_for_processing = self.prefilter.maybe_prefilter_huge(pdf_path)

            # ---------- (A) Extract structure & chunk ----------
            ds = self.heading.extract(pdf_for_processing)
            if not ds or not ds.get("sections"):
                return {"filename": pdf_path.name, "status":"failed", "error":"structure_extraction_failed"}

            raw_chunks = self.chunker.create_chunks(ds)
            if not raw_chunks:
                simp = HeadingExtractor()._simple_text(pdf_for_processing)
                if not simp:
                    return {"filename": pdf_path.name, "status":"failed", "error":"no_chunks_created"}
                raw_chunks = self.chunker.create_chunks(simp)
            if not raw_chunks:
                return {"filename": pdf_path.name, "status":"failed", "error":"no_chunks_created"}

            # ---------- (B) Save semantic chunks (idempotent) ----------
            if not self.ckpt.done(stem, "semantic_saved"):
                self.writer.save_semantic(raw_chunks, stem)
                self.ckpt.mark(stem, "semantic_saved")

            # ---------- (C) Enrich & save enriched ----------
            core, longtail = [], []
            if not self.ckpt.done(stem, "enriched_saved"):
                enriched = self.enricher.enrich_batch(raw_chunks) or []
                core     = [c for c in enriched if c.get("leishmania_relevance", 0.0) >= HIGH_RELEVANCE_THRESHOLD]
                longtail = [c for c in enriched if MIN_RELEVANCE_SCORE <= c.get("leishmania_relevance", 0.0) < HIGH_RELEVANCE_THRESHOLD]
                self.writer.save_enriched(core, longtail, stem)
                self.ckpt.mark(stem, "enriched_saved")
            else:
                # reload previously saved to compute the stats below
                core_p = Path("kaggle/working_v2/enriched_chunks/core") / f"{stem}_core_chunks.json"
                lt_p   = Path("kaggle/working_v2/enriched_chunks/longtail") / f"{stem}_longtail_chunks.json"
                if core_p.exists():
                    core = json.load(open(core_p, "r", encoding="utf-8"))
                if lt_p.exists():
                    longtail = json.load(open(lt_p, "r", encoding="utf-8"))

            # ---------- (D) Render pages (PNG) ----------
            if not self.ckpt.done(stem, "rendered"):
                try:
                    _ = render_pdf_pages_to_png(pdf_path, PAGE_RENDERS_DIR, dpi=150)
                except Exception as e:
                    # Rendering is optional; log and continue
                    logger.warning(f"Page rendering skipped for {pdf_path.name}: {e}")
                self.ckpt.mark(stem, "rendered")

            # ---------- (E) Evidence units ----------
            if not self.ckpt.done(stem, "evidence_saved"):
                try:
                    self.writer.save_evidence_units(ds, core + longtail)
                except Exception as e:
                    # Non-fatal: allow resume to continue
                    logger.warning(f"Evidence-units export failed: {e}")
                self.ckpt.mark(stem, "evidence_saved")

            # ---------- (F) Q&A ----------
            qa_count = 0
            if not self.ckpt.done(stem, "qa_saved"):
                qa = self.writer.generate_qa(core + longtail, ds)
                self.writer.save_qa(qa, stem)
                qa_count = len(qa)
                self.ckpt.mark(stem, "qa_saved")
            else:
                # We won't reload QA for stats; it's optional
                pass

            # ---------- (G) Summary ----------
            if not self.ckpt.done(stem, "summary_saved"):
                self.writer.save_summary(ds, core, longtail, stem)
                self.ckpt.mark(stem, "summary_saved")

            # Success: clear checkpoint for this doc
            self.ckpt.clear(stem)

            kept = len(core) + len(longtail)
            avg_rel = 0.0
            if kept:
                rels = [c.get("leishmania_relevance", 0.0) for c in (core + longtail)]
                avg_rel = sum(rels) / len(rels)

            return {
                "filename": pdf_path.name,
                "status": "success",
                "processed_at": datetime.now().isoformat(),
                "processing_time": time.time() - start,
                "document_structure": {
                    "total_sections": len(ds["sections"]),
                    "total_headings": len(ds["headings"]),
                    "total_characters": len(ds["full_text"])
                },
                "chunking_stats": {
                    "raw_chunks_created": len(raw_chunks),
                    "chunks_after_enrichment": kept,
                    "core_chunks": len(core),
                    "longtail_chunks": len(longtail),
                    "filtered_out": len(raw_chunks) - kept
                },
                "qa_pairs_created": qa_count,
                "avg_chunk_relevance": avg_rel,
                "high_relevance_ratio": (len(core) / kept) if kept else 0.0
            }

        except Exception as e:
            logger.error(f"Stage2 failed for {pdf_path.name}: {e}", exc_info=True)
            # Keep whatever partial checkpoint exists so next run can resume
            return {"filename": pdf_path.name, "status":"failed", "error": str(e)}

    # -------------
    # Main driver
    # -------------

    def run(self):
        if not TEXTBOOK_SOURCE_DIR.exists():
            logger.error(f"Source dir not found: {TEXTBOOK_SOURCE_DIR}")
            return

        pdfs = list(TEXTBOOK_SOURCE_DIR.rglob("*.pdf"))
        if not pdfs:
            logger.warning("No PDFs found.")
            return

        # Skip already processed if unchanged (based on saved metadata)
        already_success: Set[str] = set()
        if PROCESSED_METADATA_PATH.exists():
            try:
                df_prev = pd.read_csv(PROCESSED_METADATA_PATH)
                already_success = set(df_prev[df_prev["status"]=="success"]["filename"].tolist())
            except Exception:
                pass

        to_process = []
        for p in pdfs:
            if p.name not in already_success:
                to_process.append(p)
            elif self.hash_mgr.changed(p):
                logger.info(f"Changed since last run: {p.name}; will reprocess.")
                to_process.append(p)

        logger.info(f"Found {len(pdfs)} PDFs. Will process {len(to_process)} (new/changed).")

        # Stage 1: parallel extraction (lightweight metadata rows)
        if to_process:
            stage1 = self.stage1_parallel_extract(to_process)
            self.results.extend(stage1)
            self._save_metadata()
        else:
            stage1 = []

        # Stage 2: semantic + LLM (sequential; GPU-bound)
        successes = [r for r in stage1 if r.get("status") != "failed"]
        for i, r in enumerate(successes, 1):
            pdf_path = Path(r["filepath"])
            logger.info(f"[Stage2 {i}/{len(successes)}] {pdf_path.name}")
            res2 = self.stage2_enrich(pdf_path)
            # merge/replace record for this filename in self.results
            self.results = [x for x in self.results if x["filename"] != r["filename"]]
            merged = {**r, **res2}
            self.results.append(merged)
            self._save_metadata()

        # Print final stats
        self._print_final()

    def _print_final(self):
        df = pd.DataFrame(self.results) if self.results else pd.DataFrame()
        succ = df[df["status"]=="success"]
        fail = df[df["status"]=="failed"]
        print("\n" + "="*80)
        print("🎉 MERGED PIPELINE COMPLETE")
        print("="*80)
        print(f"✅ Successful: {len(succ)}")
        print(f"❌ Failed:     {len(fail)}")
        if not succ.empty:
            def _safe_parse(obj):
                if isinstance(obj, dict): return obj
                if isinstance(obj, str):
                    try: return literal_eval(obj)
                    except Exception: return {}
                return {}
            ch = succ["chunking_stats"].dropna().apply(_safe_parse)
            try:
                raw = sum(d.get("raw_chunks_created",0) for d in ch)
                kept= sum(d.get("chunks_after_enrichment",0) for d in ch)
                core= sum(d.get("core_chunks",0) for d in ch)
                lt  = sum(d.get("longtail_chunks",0) for d in ch)
                qa  = succ["qa_pairs_created"].fillna(0).astype(int).sum()
                print("\n📊 CHUNK STATS")
                print(f"   Raw: {raw:,}")
                print(f"   Kept: {kept:,}")
                print(f"   Core: {core:,}")
                print(f"   Longtail: {lt:,}")
                print(f"   Filtered: {raw-kept:,}")
                print(f"   Q&A: {qa:,}")
            except Exception:
                pass
        print("\n📁 Output:")
        print(f"   RAG chunks: {RAG_OUTPUT_DIR/'chunks'}")
        print(f"   Q&A:        {FINETUNE_OUTPUT_DIR/'qa_pairs'}")
        print(f"   Summaries:  {FINETUNE_OUTPUT_DIR/'summaries'}")
        print(f"   Images:     {RAG_OUTPUT_DIR/'images'}")
        print(f"   Metadata:   {PROCESSED_METADATA_PATH}")
        print("="*80 + "\n")

# ------------------
# Reset helper
# ------------------

def reset_all_outputs():
    print("⚠️ This will delete generated outputs (chunks, qa, summaries, images, metadata, hashes).")
    ok = input("Type 'reset' to proceed: ").strip().lower()
    if ok != "reset":
        print("Cancelled.")
        return
    targets = [
        RAG_OUTPUT_DIR, FINETUNE_OUTPUT_DIR,
        Path("kaggle/working_v2/semantic_chunks"),
        Path("kaggle/working_v2/enriched_chunks"),
        PROCESSED_METADATA_PATH, CHECKPOINT_PATH, FILE_HASHES_PATH
    ]
    for t in targets:
        try:
            if t.is_dir():
                shutil.rmtree(t)
                print(f"🗑️ Deleted dir: {t}")
            elif t.exists():
                t.unlink()
                print(f"🗑️ Deleted file: {t}")
        except Exception as e:
            print(f"Failed to delete {t}: {e}")
    print("✅ Reset complete.")

# ------------------
# CLI
# ------------------

def main():
    while True:
        print("\n--- Leish RAG Pipeline ---")
        print("1) Run full pipeline")
        print("2) Resume failed only")      # Move this up
        print("3) Reset all outputs (CAUTION)")
        print("4) Exit")                    # Make this option 4
        choice = input("Choose: ").strip()
        if choice == "1":
            pipe = MergedPipeline()
            pipe.run()
        elif choice == "2":                 # Change to 2
            pipe = MergedPipeline()
            pipe.run_failed_only()
        elif choice == "3":                 # Change to 3
            reset_all_outputs()
        elif choice == "4":                 # Change to 4
            print("Bye.")
            break

if __name__ == "__main__":
    main()


/home/students/Leishmania/.venv/lib/python3.11/site-packages/sentence_transformers/cross_encoder/CrossEncoder.py:11: TqdmExperimentalWarning: Using `tqdm.autonotebook.tqdm` in notebook mode. Use `tqdm.tqdm` instead to force console mode (e.g. in jupyter console)
  from tqdm.autonotebook import tqdm, trange



--- Leish RAG Pipeline ---
1) Run full pipeline
2) Resume failed only
3) Reset all outputs (CAUTION)
4) Exit


2025-09-05 15:26:07,807 - MainProcess - INFO - Loading sentence embedding model (offline preferred)...
2025-09-05 15:26:07,832 - MainProcess - INFO - Use pytorch device_name: cuda
2025-09-05 15:26:07,832 - MainProcess - INFO - Load pretrained SentenceTransformer: /data4t/hf/transformers/models--sentence-transformers--all-MiniLM-L6-v2/snapshots/c9745ed1d9f207416be6d2e6f8de32d1f16199bf
2025-09-05 15:26:07,832 - MainProcess - WARNING - No sentence-transformers model found with name /data4t/hf/transformers/models--sentence-transformers--all-MiniLM-L6-v2/snapshots/c9745ed1d9f207416be6d2e6f8de32d1f16199bf. Creating a new one with mean pooling.
2025-09-05 15:26:08,472 - MainProcess - INFO - Loading LLM (offline preferred): mistralai/Mistral-7B-Instruct-v0.1
2025-09-05 15:26:08,481 - MainProcess - INFO - LLM loading from: /data4t/hf/transformers/models--mistralai--Mistral-7B-Instruct-v0.1/snapshots/ec5deb64f2c6e6fa90c1abf74a91d5c93a9669ca
2025-09-05 15:26:08,863 - MainProcess - INFO - We will 

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Device set to use cuda:0
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
2025-09-05 15:26:47,679 - MainProcess - ERROR - Source dir not found: data/all_leishmania_sources



--- Leish RAG Pipeline ---
1) Run full pipeline
2) Resume failed only
3) Reset all outputs (CAUTION)
4) Exit

--- Leish RAG Pipeline ---
1) Run full pipeline
2) Resume failed only
3) Reset all outputs (CAUTION)
4) Exit

--- Leish RAG Pipeline ---
1) Run full pipeline
2) Resume failed only
3) Reset all outputs (CAUTION)
4) Exit


KeyboardInterrupt: Interrupted by user